# Movie store Exploration

Data setup with Sakila film dataset 

In [16]:
import duckdb
from pathlib import Path

duckdb_path = "data/sakila.duckdb"
Path(duckdb_path).unlink(missing_ok=True)

with duckdb.connect(duckdb_path) as conn, open("sql/load_sakila.sql") as ingest_script:
    conn.sql(ingest_script.read())

    description = conn.sql("DESC;").df()
    films = conn.sql("FROM film;").df() #films table are now loaded into a pandas dataframe films

films.head(3)

,film_id,title,description,release_year,language_id,original_language_id,rental_duration,rental_rate,length,replacement_cost,rating,special_features,last_update
0,1,ACADEMY DINOSAUR,A Epic Drama of a Feminist And a Mad Scientist...,2006,1,<NA>,6,0.99,86,20.99,PG,"Deleted Scenes,Behind the Scenes",2021-03-06 15:52:00
1,2,ACE GOLDFINGER,A Astounding Epistle of a Database Administrat...,2006,1,<NA>,3,4.99,48,12.99,G,"Trailers,Deleted Scenes",2021-03-06 15:52:00
2,3,ADAPTATION HOLES,A Astounding Reflection of a Lumberjack And a ...,2006,1,<NA>,7,2.99,50,18.99,NC-17,"Trailers,Deleted Scenes",2021-03-06 15:52:00


Which movies are longer than 3 hours (180 minutes), lets explore the title and its length

In [17]:

films['length'].head(3)

0    86
1    48
2    50
Name: length, dtype: int64

In [18]:
long_films = films[films["length"] > 180]
long_films.head()

,film_id,title,description,release_year,language_id,original_language_id,rental_duration,rental_rate,length,replacement_cost,rating,special_features,last_update
23,24,ANALYZE HOOSIERS,A Thoughtful Display of a Explorer And a Pastr...,2006,1,<NA>,6,2.99,181,19.99,R,"Trailers,Behind the Scenes",2021-03-06 15:52:00
49,50,BAKED CLEOPATRA,A Stunning Drama of a Forensic Psychologist An...,2006,1,<NA>,3,2.99,182,20.99,G,"Commentaries,Behind the Scenes",2021-03-06 15:52:01
127,128,CATCH AMISTAD,A Boring Reflection of a Lumberjack And a Femi...,2006,1,<NA>,7,0.99,183,10.99,G,"Trailers,Behind the Scenes",2021-03-06 15:52:01
140,141,CHICAGO NORTH,A Fateful Yarn of a Mad Cow And a Waitress who...,2006,1,<NA>,6,4.99,185,11.99,PG-13,"Deleted Scenes,Behind the Scenes",2021-03-06 15:52:01
179,180,CONSPIRACY SPIRIT,A Awe-Inspiring Story of a Student And a Frisb...,2006,1,<NA>,4,2.99,184,27.99,PG-13,"Trailers,Commentaries",2021-03-06 15:52:02


In [19]:
result = long_films[["title", "length"]]
result = result.sort_values("length", ascending=False)
print(result)

                  title  length
689        POND SEATTLE     185
990        WORST BANGER     185
140       CHICAGO NORTH     185
181      CONTROL ANTHEM     185
816  SOLDIERS EVOLUTION     185
211      DARN FORRESTER     185
348         GANGS PRIDE     185
608       MUSCLE BRIGHT     185
425           HOME PITY     185
871   SWEET BROTHERHOOD     185
819      SONS INTERVIEW     184
812     SMOOCHY CONTROL     184
498      KING EVOLUTION     184
596     MOONWALKER FOOL     184
820      SORORITY QUEEN     184
197    CRYSTAL BREAKING     184
179   CONSPIRACY SPIRIT     184
885      THEORY MERMAID     184
972           WIFE TURN     183
766       SCALAWAG DUCK     183
995      YOUNG LANGUAGE     183
127       CATCH AMISTAD     183
339      FRONTIER CABIN     183
49      BAKED CLEOPATRA     182
720          REDS POCUS     182
764         SATURN NAME     182
590       MONSOON CAUSE     182
773      SEARCHERS WAIT     182
718       RECORDS ZORRO     182
434     HOTEL HAPPINESS     181
973     

Films with LOVE in their title

In [20]:
result = films.loc[
    films["title"].str.contains("love", case=False, na=False),
    ["title", "rating", "length", "description"]
]
print(result.sort_values("title"))

                  title rating  length  \
373       GRAFFITI LOVE     PG     117   
447          IDAHO LOVE  PG-13     172   
448      IDENTITY LOVER  PG-13     119   
457         INDIAN LOVE  NC-17     135   
510       LAWRENCE LOVE  NC-17     175   
534       LOVE SUICIDES      R     181   
535       LOVELY JINGLE     PG      65   
536        LOVER TRUMAN      G      75   
537    LOVERBOY ATTACKS  PG-13     162   
851  STRANGELOVE DESIRE  NC-17     103   

                                           description  
373  A Unbelieveable Epistle of a Sumo Wrestler And...  
447  A Fast-Paced Drama of a Student And a Crocodil...  
448  A Boring Tale of a Composer And a Mad Cow who ...  
457  A Insightful Saga of a Mad Scientist And a Mad...  
510  A Fanciful Yarn of a Database Administrator An...  
534  A Brilliant Panorama of a Hunter And a Explore...  
535  A Fanciful Yarn of a Crocodile And a Forensic ...  
536  A Emotional Yarn of a Robot And a Boy who must...  
537  A Boring Story of a

Explorative data analysis 

In [21]:
dfs = {}

with duckdb.connect(duckdb_path) as conn:
    for name in description["name"]:
        dfs[name] = conn.sql(f"FROM {name};").df()

dfs.keys()

dict_keys(['actor', 'address', 'category', 'city', 'country', 'customer', 'customer_list', 'film', 'film_actor', 'film_category', 'film_list', 'film_text', 'inventory', 'language', 'payment', 'rental', 'sales_by_film_category', 'sales_by_store', 'staff', 'staff_list', 'store'])

In [22]:
film_names = ("film", "film_actor", "film_category", "actor", "category")

for film_name in film_names:
    duckdb.register(film_name, dfs[film_name])

duckdb.sql("desc;").df()

,database,schema,name,column_names,column_types,temporary
0,temp,main,actor,"[actor_id, first_name, last_name, last_update]","[DOUBLE, VARCHAR, VARCHAR, TIMESTAMP]",True
1,temp,main,category,"[category_id, name, last_update]","[BIGINT, VARCHAR, TIMESTAMP]",True
2,temp,main,film,"[film_id, title, description, release_year, la...","[BIGINT, VARCHAR, VARCHAR, VARCHAR, BIGINT, BI...",True
3,temp,main,film_actor,"[actor_id, film_id, last_update]","[BIGINT, BIGINT, TIMESTAMP]",True
4,temp,main,film_category,"[film_id, category_id, last_update]","[BIGINT, BIGINT, TIMESTAMP]",True


In [23]:
films_joined = duckdb.sql("""
    SELECT
        a.first_name || ' ' || a.last_name AS actor,
        a.actor_id::INT AS actor_id,
        f.title,
        f.description,
        f.release_year,
        f.rental_duration,
        f.rating,
        c.name AS category
    FROM film f
        LEFT JOIN film_actor fa ON f.film_id = fa.film_id
        LEFT JOIN actor a ON a.actor_id = fa.actor_id
        LEFT JOIN film_category fc ON fc.film_id = f.film_id      
        LEFT JOIN category c ON fc.category_id = c.category_id      
""").df()

films_joined.head(2)

,actor,actor_id,title,description,release_year,rental_duration,rating,category
0,PENELOPE GUINESS,1,ACADEMY DINOSAUR,A Epic Drama of a Feminist And a Mad Scientist...,2006,6,PG,Documentary
1,PENELOPE GUINESS,1,ANACONDA CONFESSIONS,A Lacklusture Display of a Dentist And a Denti...,2006,3,R,Animation
